
---
## 📡 GDELT Lexical Panel Builder

**Purpose**  
Aggregates the raw GDELT event database into a monthly dyadic panel containing conflict/cooperation counts, total event volume, and the average Goldstein score. This panel is later used as one of the sources in the composite GPR index.

**Input**  
- GDELT daily export files ([GDELT DOWNLOADS ](http://data.gdeltproject.org/events/)).  
  The notebook downloads them on‑the‑fly (or reads them from a local cache).

**Output**  
- `data/03_nlp/gdelt/gdelt_birectional_lexical_panel.csv` – monthly panel with columns:  
  `date`, `dyad`, `total_volume`, `url_conflict_hits`, `url_cooperation_hits`, `cameo_conflict_hits`, `cameo_cooperation_hits`, `goldstein_mean`.

**🛠️ To avoid path errors (run this notebook from anywhere)**  

The notebook already uses **relative paths** (e.g. `GDELT_DIR = BASE / "data" / "03_nlp" / "gdelt"`).  
Make sure you open the notebook from the **project root** (the main root folder containing `notebooks/`, `data/`, etc.).  

If you need to change the location of the output directory, locate the cell that defines `BASE` (typically near the top) and modify it:

```python
BASE = Path.cwd().parent   
DATA = BASE / "data" / "03_nlp"
```
No absolute Windows paths are hard‑coded; the notebook uses Path objects and relative navigation.

**Dependencies:** Requires `pandas`, `numpy`, `requests`, `tqdm` ... Install them with `pip install -r requirements.txt` 

---

### 🔧 General advice for both notebooks

- **Hard‑coded strings like `C:\Users\HP\...`** only appear in printed output (e.g. `print(f"ROOT → {ROOT}")`) and they do **not** affect where files are read/written. The code uses `Path` relative paths.
- **Run the notebook from the project root** – e.g. if your repo structure is:
macro-geopolitics/
├── notebooks/
│ └── gdelt_data_acquisition/
│ ├── 01_gdelt_lexical_panel.ipynb
│ └── 02_gdelt_url_corpus.ipynb
├── data/
├── outputs/
└── ...
then `cd macro-geopolitics` and launch Jupyter from there. The relative paths will resolve correctly.
- **If you must use a different base folder**, define a variable `PROJECT_ROOT = Path("/your/absolute/path")` at the very top of the notebook after removing the ones I've defined, and then build all other paths as `PROJECT_ROOT / "data" / ...`.
___
___

In [ ]:
import pandas as pd

# Load the file being continuously updated
df = pd.read_csv("data_nlp/gdelt_birectional_lexical_panel.csv")

print("--- FILE SHAPE ---")
print(f"Total rows written so far: {len(df)}")

print("\n--- SAMPLE ENTRIES ---")
print(df.head(11))  # Look at the first month's 11 dyads

In [ ]:
from __future__ import annotations
import io
import re
import zipfile
from datetime import datetime
from pathlib import Path
import pandas as pd
import requests
from tqdm import tqdm

OUTPUT_DIR = Path("data_nlp")
OUTPUT_FILE = OUTPUT_DIR / "gdelt_birectional_lexical_panel.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GDELT_BASE = "http://data.gdeltproject.org/events/"

DYADS = [
    ('CHN', 'USA'), ('CHN', 'JPN'), ('CHN', 'AUS'),
    ('CHN', 'FRA'), ('CHN', 'DEU'), ('CHN', 'GBR'),
    ('CHN', 'RUS'), ('CHN', 'IND'), ('CHN', 'IDN'),
    ('CHN', 'PAK'), ('CHN', 'VNM')
]

COL = {
    "sqldate": 1, "actor1_code": 5, "actor2_code": 15,
    "event_code": 25, "event_root": 26, "goldstein": 30, "sourceurl": 57,
}

LEXICAL_CONFLICT = [
    "war", "warfare", "military", "armed conflict", "conflict", "terror", "terrorism",
    "attack", "invasion", "invade", "battle", "troops", "sanction", "sanctions",
    "embargo", "hostage", "crisis", "hostile", "retaliation", "trade war", "tariff", "tariffs"
]
LEXICAL_COOPERATION = [
    "cooperation", "cooperate", "diplomacy", "diplomatic", "peace", "treaty", "accord",
    "alliance", "rapprochement", "de-escalation", "normalization", "summit", "partnership"
]

CAMEO_CONFLICT_ROOTS = {14, 15, 17, 18, 19, 20}
CAMEO_COOPERATION_ROOTS = {1, 2, 3, 4, 5, 8}

def compile_patterns():
    conf_pat = re.compile(r"\b(" + "|".join(re.escape(w) for w in LEXICAL_CONFLICT) + r")\b", re.I)
    coop_pat = re.compile(r"\b(" + "|".join(re.escape(w) for w in LEXICAL_COOPERATION) + r")\b", re.I)
    return conf_pat, coop_pat

def download_file(session: requests.Session, url: str) -> bytes | None:
    try:
        response = session.get(url, timeout=30)
        if response.status_code == 200:
            return response.content
    except Exception:
        pass
    return None

def extract_dyad_events(df: pd.DataFrame, a1: str, a2: str, target_month_str: str, check_date: bool) -> pd.DataFrame:
    c1, c2 = COL["actor1_code"], COL["actor2_code"]
    mask = ((df[c1] == a1) & (df[c2] == a2)) | ((df[c1] == a2) & (df[c2] == a1))
    sub = df.loc[mask]
    if sub.empty:
        return pd.DataFrame()
    if check_date:
        date_col = COL["sqldate"]
        sub = sub[sub[date_col].fillna("").astype(str).str.startswith(target_month_str)]
        if sub.empty:
            return pd.DataFrame()
    out = pd.DataFrame()
    out["event_root"] = pd.to_numeric(sub[COL["event_root"]], errors="coerce")
    out["goldstein"] = pd.to_numeric(sub[COL["goldstein"]], errors="coerce")
    if sub.shape[1] > COL["sourceurl"]:
        out["sourceurl"] = sub[COL["sourceurl"]].fillna("").astype(str).str.strip()
    else:
        out["sourceurl"] = ""
    return out

def label_nlp_metrics(df: pd.DataFrame, conf_pat: re.Pattern, coop_pat: re.Pattern) -> dict:
    if df.empty:
        return {
            "total_volume": 0, "url_conflict_hits": 0, "url_cooperation_hits": 0,
            "cameo_conflict_hits": 0, "cameo_cooperation_hits": 0, "goldstein_mean": None
        }
    has_url = df["sourceurl"].str.startswith("http", na=False)
    cameo_conflict = int(df["event_root"].isin(CAMEO_CONFLICT_ROOTS).sum())
    cameo_cooperation = int(df["event_root"].isin(CAMEO_COOPERATION_ROOTS).sum())
    url_conflict = 0
    url_cooperation = 0
    if has_url.any():
        url_strings = df.loc[has_url, "sourceurl"]
        url_conflict = int(url_strings.map(lambda text: bool(conf_pat.search(text))).sum())
        url_cooperation = int(url_strings.map(lambda text: bool(coop_pat.search(text))).sum())
    return {
        "total_volume": len(df),
        "url_conflict_hits": url_conflict,
        "url_cooperation_hits": url_cooperation,
        "cameo_conflict_hits": cameo_conflict,
        "cameo_cooperation_hits": cameo_cooperation,
        "goldstein_mean": df["goldstein"].mean()
    }

def process_zip(blob: bytes, conf_pat: re.Pattern, coop_pat: re.Pattern,
                target_month_str: str, check_date: bool) -> list[dict]:
    records = []
    try:
        with zipfile.ZipFile(io.BytesIO(blob)) as zf:
            for name in zf.namelist():
                if not name.lower().endswith(".csv"):
                    continue
                with zf.open(name) as f:
                    df = pd.read_csv(f, sep="\t", header=None, dtype=str, low_memory=False)
                    for (a1, a2) in DYADS:
                        dyad_name = f"{a1}-{a2}"
                        df_dyad = extract_dyad_events(df, a1, a2, target_month_str, check_date)
                        if not df_dyad.empty:
                            metrics = label_nlp_metrics(df_dyad, conf_pat, coop_pat)
                            metrics["dyad"] = dyad_name
                            records.append(metrics)
    except Exception:
        pass
    return records

def process_daily_month(session: requests.Session, month: datetime, conf_pat, coop_pat) -> list[dict]:
    target_month_str = month.strftime("%Y%m")
    days = pd.date_range(start=month, end=month + pd.offsets.MonthEnd(0), freq='D')
    month_records = []
    for day in days:
        day_str = day.strftime("%Y%m%d")
        url = f"{GDELT_BASE}{day_str}.export.CSV.zip"
        blob = download_file(session, url)
        if blob:
            day_records = process_zip(blob, conf_pat, coop_pat, target_month_str, check_date=False)
            month_records.extend(day_records)
    return month_records

def process_monthly_zip(session: requests.Session, month: datetime, conf_pat, coop_pat) -> list[dict]:
    target_month_str = month.strftime("%Y%m")
    url = f"{GDELT_BASE}{target_month_str}.zip"
    blob = download_file(session, url)
    if blob:
        return process_zip(blob, conf_pat, coop_pat, target_month_str, check_date=False)
    return []

def main():
    print("=" * 70)
    print("GDELT PANEL GENERATOR – ROBUST DATETIME RESUME")
    print("=" * 70)

    latest_existing_date = None
    if OUTPUT_FILE.exists():
        try:
            df_existing = pd.read_csv(OUTPUT_FILE)
            if not df_existing.empty and "date" in df_existing.columns:
                raw_dates = df_existing["date"].dropna().unique()
                parsed_dates = [pd.to_datetime(d.strip()) for d in raw_dates if d.strip()]
                if parsed_dates:
                    latest_existing_date = max(parsed_dates)
                    print(f"Found existing logs up to: {latest_existing_date.strftime('%Y-%m-%d')}")
        except Exception as e:
            print(f"⚠️ Error parsing output file metadata: {e}")

    conf_pat, coop_pat = compile_patterns()
    
    # 2006-2022 Range Setup
    months_2006_on = pd.date_range("2006-01-01", "2022-02-01", freq="MS")
    
    # Enforce resuming strictly *after* the maximum date found in your CSV
    if latest_existing_date is not None:
        months_to_do = [m for m in months_2006_on if m > latest_existing_date]
    else:
        months_to_do = list(months_2006_on)

    if not months_to_do:
        print("\n✅ All months up to February 2022 are completely processed. Exiting.")
        return

    print(f"\nProcessing {len(months_to_do)} outstanding months from {months_to_do[0].strftime('%Y-%m')} to {months_to_do[-1].strftime('%Y-%m')}")
    
    with requests.Session() as session:
        for month in tqdm(months_to_do, desc="2006-2022 timeline"):
            date_key = month.strftime("%Y-%m-%d")
            if month < datetime(2013, 4, 1):
                records = process_monthly_zip(session, month, conf_pat, coop_pat)
            else:
                records = process_daily_month(session, month, conf_pat, coop_pat)

            if records:
                df_month = pd.DataFrame(records)
                summary = df_month.groupby("dyad").agg({
                    "total_volume": "sum", "url_conflict_hits": "sum", "url_cooperation_hits": "sum",
                    "cameo_conflict_hits": "sum", "cameo_cooperation_hits": "sum", "goldstein_mean": "mean"
                }).reset_index()
                summary["date"] = date_key
                summary = summary[["date", "dyad", "total_volume", "url_conflict_hits", "url_cooperation_hits", "cameo_conflict_hits", "cameo_cooperation_hits", "goldstein_mean"]]
                summary.to_csv(OUTPUT_FILE, mode='a', header=not OUTPUT_FILE.exists(), index=False)
                tqdm.write(f"  ✓ Saved panel updates for: {date_key}")

    print(f"\n🚀 Pipeline finished! Summary output locked into: {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

GDELT PANEL GENERATOR – ROBUST DATETIME RESUME
Found existing logs up to: 2020-08-01

Processing 18 outstanding months from 2020-09 to 2022-02


2006-2022 timeline:   6%|▌         | 1/18 [11:19<3:12:38, 679.89s/it]

  ✓ Saved panel updates for: 2020-09-01


2006-2022 timeline:  11%|█         | 2/18 [20:48<2:43:52, 614.54s/it]

  ✓ Saved panel updates for: 2020-10-01


...

  ✓ Saved panel updates for: 2021-12-01


2006-2022 timeline:  94%|█████████▍| 17/18 [2:41:15<08:32, 512.32s/it]

  ✓ Saved panel updates for: 2022-01-01


2006-2022 timeline: 100%|██████████| 18/18 [2:48:10<00:00, 560.59s/it]

  ✓ Saved panel updates for: 2022-02-01

🚀 Pipeline finished! Summary output locked into: data_nlp\gdelt_birectional_lexical_panel.csv


In [ ]:
import pandas as pd
from datetime import datetime

# Load your existing CSV
df = pd.read_csv("data_nlp/gdelt_birectional_lexical_panel.csv")

# Get all dates present
existing_dates = set(df['date'].unique())
print(f"Total months in CSV: {len(existing_dates)}")

# Generate all expected months from 1990-01 to 2005-12
expected_months = pd.date_range("1990-01-01", "2005-12-01", freq="MS").strftime("%Y-%m-%d")
missing = [m for m in expected_months if m not in existing_dates]
print(f"Missing months in 1990-2005: {len(missing)}")
if missing:
    print("First 10 missing:", missing[:10])

Total months in CSV: 380
Missing months in 1990-2005: 0


In [ ]:
from pathlib import Path
import pandas as pd

def audit_gdelt_panel():
    csv_path = Path("data_nlp/gdelt_birectional_lexical_panel.csv")
    
    # 1. Define Expected Target Ranges and Dimensions
    expected_months = pd.date_range(start="1990-01-01", end="2022-02-01", freq="MS")
    expected_dates_str = set(expected_months.strftime("%Y-%m-%d"))
    
    expected_dyads = {
        'CHN-USA', 'CHN-JPN', 'CHN-AUS', 'CHN-FRA', 'CHN-DEU', 
        'CHN-GBR', 'CHN-RUS', 'CHN-IND', 'CHN-IDN', 'CHN-PAK', 'CHN-VNM'
    }
    
    total_expected_rows = len(expected_dates_str) * len(expected_dyads)
    
    print("=" * 70)
    print("GDELT DATA PANEL INTEGRITY AUDIT")
    print("=" * 70)
    
    if not csv_path.exists():
        print(f"❌ Error: Target file not found at {csv_path}")
        return

    # 2. Load and Parse Output Data
    df = pd.read_csv(csv_path)
    
    if df.empty:
        print("❌ Error: The data file exists but is completely empty.")
        return
        
    if "date" not in df.columns or "dyad" not in df.columns:
        print("❌ Error: Missing required structural columns ('date' or 'dyad').")
        return

    # Standardize string formatting to eliminate whitespace issues
    df["date"] = df["date"].astype(str).str.strip()
    df["dyad"] = df["dyad"].astype(str).str.strip()
    
    actual_dates = set(df["date"].unique())
    actual_dyads = set(df["dyad"].unique())
    
    # 3. Structural Gaps Analysis
    missing_entire_months = sorted(list(expected_dates_str - actual_dates))
    unexpected_dates = sorted(list(actual_dates - expected_dates_str))
    missing_global_dyads = expected_dyads - actual_dyads
    
    print(f"• Total Rows Found: {len(df)} (Expected: {total_expected_rows})")
    print(f"• Unique Months Represented: {len(actual_dates)} / {len(expected_dates_str)}")
    
    # Report Complete Gaps
    if missing_entire_months:
        print(f"\n❌ CRITICAL CRASH GAPS: Missing {len(missing_entire_months)} months entirely:")
        # Group missing months cleanly by year for readable output logs
        by_year = {}
        for d in missing_entire_months:
            by_year.setdefault(d[:4], []).append(d[5:7])
        for year, months in sorted(by_year.items()):
            print(f"  - Year {year} Months: {', '.join(months)}")
    else:
        print("\n✅ Timeline Continuity: No months are missing entirely from the panel range.")

    if missing_global_dyads:
        print(f"❌ Structural Flaw: Dyads entirely missing globally: {missing_global_dyads}")

    # 4. Deep Combinatorial Row-Level Audit
    # Tracks down months that saved data, but missed specific dyads due to partial pipeline failures
    print("\n• Scanning individual rows for missing cross-sections...")
    
    # Build a rapid lookup set of tuples (date, dyad)
    present_pairs = set(zip(df["date"], df["dyad"]))
    
    partial_month_issues = {}
    for dt in sorted(list(expected_dates_str)):
        if dt in missing_entire_months:
            continue  # Already flagged above
        missing_from_this_month = []
        for dy in expected_dyads:
            if (dt, dy) not in present_pairs:
                missing_from_this_month.append(dy)
        if missing_from_this_month:
            partial_month_issues[dt] = missing_from_this_month

    if partial_month_issues:
        print(f"❌ PARTIAL MONTH COMPLETENESS ISSUES: Found {len(partial_month_issues)} months missing specific dyads:")
        for dt, dyads in sorted(partial_month_issues.items()):
            print(f"  - {dt} is missing {len(dyads)} dyad(s): {', '.join(dyads)}")
    else:
        print("✅ Cross-Sectional Density: Every recorded month contains all 11 required country dyads.")

    # 5. Summary Verdict
    print("\n" + "=" * 70)
    if not missing_entire_months and not partial_month_issues and not missing_global_dyads:
        print("🎉 SUCCESS: Data panel structure is completely dense, uniform, and pristine!")
    else:
        print("⚠️ AUDIT FAILED: Structural gaps identified. Review the logs above to identify data missingness.")
    print("=" * 70)

if __name__ == "__main__":
    audit_gdelt_panel()

GDELT DATA PANEL INTEGRITY AUDIT
• Total Rows Found: 4109 (Expected: 4246)
• Unique Months Represented: 380 / 386

❌ CRITICAL CRASH GAPS: Missing 6 months entirely:
  - Year 2013 Months: 03, 04, 05, 06, 07, 08

• Scanning individual rows for missing cross-sections...
❌ PARTIAL MONTH COMPLETENESS ISSUES: Found 47 months missing specific dyads:
  - 1990-01-01 is missing 2 dyad(s): CHN-IDN, CHN-IND
  - 1990-02-01 is missing 4 dyad(s): CHN-DEU, CHN-VNM, CHN-IND, CHN-AUS
  - 1990-04-01 is missing 2 dyad(s): CHN-IDN, CHN-DEU
  - 1990-07-01 is missing 1 dyad(s): CHN-IND
  - 1990-08-01 is missing 2 dyad(s): CHN-DEU, CHN-IND
  - 1990-09-01 is missing 2 dyad(s): CHN-DEU, CHN-IND
  - 1990-10-01 is missing 1 dyad(s): CHN-IND
  - 1990-11-01 is missing 3 dyad(s): CHN-IND, CHN-AUS, CHN-PAK
  - 1990-12-01 is missing 2 dyad(s): CHN-IND, CHN-AUS
  - 1991-01-01 is missing 3 dyad(s): CHN-IND, CHN-AUS, CHN-PAK
  - 1991-02-01 is missing 1 dyad(s): CHN-DEU
  - 1991-04-01 is missing 2 dyad(s): CHN-IDN, CHN-IN

In [ ]:
from __future__ import annotations
import io
import re
import zipfile
from datetime import datetime
from pathlib import Path
import pandas as pd
import requests
from tqdm import tqdm

OUTPUT_DIR = Path("data_nlp")
OUTPUT_FILE = OUTPUT_DIR / "gdelt_birectional_lexical_panel.csv"

GDELT_BASE = "http://data.gdeltproject.org/events/"

DYADS = [
    ('CHN', 'USA'), ('CHN', 'JPN'), ('CHN', 'AUS'),
    ('CHN', 'FRA'), ('CHN', 'DEU'), ('CHN', 'GBR'),
    ('CHN', 'RUS'), ('CHN', 'IND'), ('CHN', 'IDN'),
    ('CHN', 'PAK'), ('CHN', 'VNM')
]

COL = {
    "sqldate": 1, "actor1_code": 5, "actor2_code": 15,
    "event_code": 25, "event_root": 26, "goldstein": 30, "sourceurl": 57,
}

LEXICAL_CONFLICT = [
    "war", "warfare", "military", "armed conflict", "conflict", "terror", "terrorism",
    "attack", "invasion", "invade", "battle", "troops", "sanction", "sanctions",
    "embargo", "hostage", "crisis", "hostile", "retaliation", "trade war", "tariff", "tariffs"
]
LEXICAL_COOPERATION = [
    "cooperation", "cooperate", "diplomacy", "diplomatic", "peace", "treaty", "accord",
    "alliance", "rapprochement", "de-escalation", "normalization", "summit", "partnership"
]

CAMEO_CONFLICT_ROOTS = {14, 15, 17, 18, 19, 20}
CAMEO_COOPERATION_ROOTS = {1, 2, 3, 4, 5, 8}

def compile_patterns():
    conf_pat = re.compile(r"\b(" + "|".join(re.escape(w) for w in LEXICAL_CONFLICT) + r")\b", re.I)
    coop_pat = re.compile(r"\b(" + "|".join(re.escape(w) for w in LEXICAL_COOPERATION) + r")\b", re.I)
    return conf_pat, coop_pat

def download_file(session: requests.Session, url: str) -> bytes | None:
    try:
        response = session.get(url, timeout=30)
        if response.status_code == 200:
            return response.content
    except Exception:
        pass
    return None

def extract_dyad_events(df: pd.DataFrame, a1: str, a2: str, target_month_str: str, check_date: bool) -> pd.DataFrame:
    c1, c2 = COL["actor1_code"], COL["actor2_code"]
    mask = ((df[c1] == a1) & (df[c2] == a2)) | ((df[c1] == a2) & (df[c2] == a1))
    sub = df.loc[mask]
    if sub.empty:
        return pd.DataFrame()
    if check_date:
        date_col = COL["sqldate"]
        sub = sub[sub[date_col].fillna("").astype(str).str.startswith(target_month_str)]
        if sub.empty:
            return pd.DataFrame()
    out = pd.DataFrame()
    out["event_root"] = pd.to_numeric(sub[COL["event_root"]], errors="coerce")
    out["goldstein"] = pd.to_numeric(sub[COL["goldstein"]], errors="coerce")
    if sub.shape[1] > COL["sourceurl"]:
        out["sourceurl"] = sub[COL["sourceurl"]].fillna("").astype(str).str.strip()
    else:
        out["sourceurl"] = ""
    return out

def label_nlp_metrics(df: pd.DataFrame, conf_pat: re.Pattern, coop_pat: re.Pattern) -> dict:
    if df.empty:
        return {
            "total_volume": 0, "url_conflict_hits": 0, "url_cooperation_hits": 0,
            "cameo_conflict_hits": 0, "cameo_cooperation_hits": 0, "goldstein_mean": None
        }
    has_url = df["sourceurl"].str.startswith("http", na=False)
    cameo_conflict = int(df["event_root"].isin(CAMEO_CONFLICT_ROOTS).sum())
    cameo_cooperation = int(df["event_root"].isin(CAMEO_COOPERATION_ROOTS).sum())
    url_conflict = 0
    url_cooperation = 0
    if has_url.any():
        url_strings = df.loc[has_url, "sourceurl"]
        url_conflict = int(url_strings.map(lambda text: bool(conf_pat.search(text))).sum())
        url_cooperation = int(url_strings.map(lambda text: bool(coop_pat.search(text))).sum())
    return {
        "total_volume": len(df),
        "url_conflict_hits": url_conflict,
        "url_cooperation_hits": url_cooperation,
        "cameo_conflict_hits": cameo_conflict,
        "cameo_cooperation_hits": cameo_cooperation,
        "goldstein_mean": df["goldstein"].mean()
    }

def process_zip(blob: bytes, conf_pat: re.Pattern, coop_pat: re.Pattern,
                target_month_str: str, check_date: bool) -> list[dict]:
    records = []
    try:
        with zipfile.ZipFile(io.BytesIO(blob)) as zf:
            for name in zf.namelist():
                if not name.lower().endswith(".csv"):
                    continue
                with zf.open(name) as f:
                    df = pd.read_csv(f, sep="\t", header=None, dtype=str, low_memory=False)
                    for (a1, a2) in DYADS:
                        dyad_name = f"{a1}-{a2}"
                        df_dyad = extract_dyad_events(df, a1, a2, target_month_str, check_date)
                        if not df_dyad.empty:
                            metrics = label_nlp_metrics(df_dyad, conf_pat, coop_pat)
                            metrics["dyad"] = dyad_name
                            records.append(metrics)
    except Exception:
        pass
    return records

def process_daily_month(session: requests.Session, month: datetime, conf_pat, coop_pat) -> list[dict]:
    target_month_str = month.strftime("%Y%m")
    days = pd.date_range(start=month, end=month + pd.offsets.MonthEnd(0), freq='D')
    month_records = []
    for day in days:
        day_str = day.strftime("%Y%m%d")
        url = f"{GDELT_BASE}{day_str}.export.CSV.zip"
        blob = download_file(session, url)
        if blob:
            day_records = process_zip(blob, conf_pat, coop_pat, target_month_str, check_date=False)
            month_records.extend(day_records)
    return month_records

def process_monthly_zip(session: requests.Session, month: datetime, conf_pat, coop_pat) -> list[dict]:
    target_month_str = month.strftime("%Y%m")
    url = f"{GDELT_BASE}{target_month_str}.zip"
    blob = download_file(session, url)
    if blob:
        return process_zip(blob, conf_pat, coop_pat, target_month_str, check_date=False)
    return []

def main():
    print("=" * 70)
    print("GDELT TARGETED RECOVERY PATCHER")
    print("=" * 70)

    if not OUTPUT_FILE.exists():
        print("❌ Source panel file not found. Run your main script baseline first.")
        return

    df_panel = pd.read_csv(OUTPUT_FILE)
    df_panel["date"] = df_panel["date"].astype(str).str.strip()
    df_panel["dyad"] = df_panel["dyad"].astype(str).str.strip()

    conf_pat, coop_pat = compile_patterns()

    # Define targeted gaps from the audit output logs
    missing_entirely = ["2013-03-01", "2013-04-01", "2013-05-01", "2013-06-01", "2013-07-01", "2013-08-01"]
    
    partial_gaps = {
        "1990-01-01": ["CHN-IDN", "CHN-IND"],
        "1990-02-01": ["CHN-DEU", "CHN-VNM", "CHN-IND", "CHN-AUS"],
        "1990-04-01": ["CHN-IDN", "CHN-DEU"],
        "1990-07-01": ["CHN-IND"],
        "1990-08-01": ["CHN-DEU", "CHN-IND"],
        "1990-09-01": ["CHN-DEU", "CHN-IND"],
        "1990-10-01": ["CHN-IND"],
        "1990-11-01": ["CHN-IND", "CHN-AUS", "CHN-PAK"],
        "1990-12-01": ["CHN-IND", "CHN-AUS"],
        "1991-01-01": ["CHN-IND", "CHN-AUS", "CHN-PAK"],
        "1991-02-01": ["CHN-DEU"],
        "1991-04-01": ["CHN-IDN", "CHN-IND"],
        "1991-06-01": ["CHN-IND"],
        "1991-07-01": ["CHN-IND"],
        "1991-08-01": ["CHN-IDN", "CHN-IND"],
        "1991-09-01": ["CHN-IND"],
        "1991-11-01": ["CHN-IND"],
        "1991-12-01": ["CHN-DEU", "CHN-AUS"],
        "1992-01-01": ["CHN-IND"],
        "1992-02-01": ["CHN-DEU", "CHN-IND", "CHN-AUS"],
        "1992-03-01": ["CHN-IND"],
        "1992-06-01": ["CHN-IND"],
        "1992-07-01": ["CHN-IND", "CHN-AUS"],
        "1992-08-01": ["CHN-DEU"],
        "1992-09-01": ["CHN-IND"],
        "1992-11-01": ["CHN-IDN", "CHN-IND"],
        "1993-01-01": ["CHN-IDN", "CHN-PAK"],
        "1993-02-01": ["CHN-DEU"],
        "1993-03-01": ["CHN-IND"],
        "1993-04-01": ["CHN-IND", "CHN-PAK"],
        "1993-06-01": ["CHN-IDN", "CHN-DEU"],
        "1993-07-01": ["CHN-IND"],
        "1993-08-01": ["CHN-DEU"],
        "1993-09-01": ["CHN-DEU"],
        "1993-10-01": ["CHN-IND"],
        "1993-11-01": ["CHN-IDN"],
        "1993-12-01": ["CHN-IDN", "CHN-DEU", "CHN-AUS"],
        "1994-05-01": ["CHN-PAK"],
        "1995-07-01": ["CHN-IND"],
        "1995-09-01": ["CHN-DEU"],
        "1995-10-01": ["CHN-IND"],
        "1996-01-01": ["CHN-IND"],
        "1996-02-01": ["CHN-IND"],
        "1996-04-01": ["CHN-IND"],
        "1996-06-01": ["CHN-IND"],
        "1996-10-01": ["CHN-IND"],
        "1997-10-01": ["CHN-IND"]
    }

    recovered_records = []

    with requests.Session() as session:
        # Phase 1: Re-fetch the 6 missing 2013 months completely
        print("⚡ Phase 1: Executing complete recovery of 2013 transitional months...")
        for date_str in tqdm(missing_entirely, desc="2013 Recovery"):
            month_dt = pd.to_datetime(date_str)
            # 2013-03 is legacy monthly file structure
            if month_dt < datetime(2013, 4, 1):
                records = process_monthly_zip(session, month_dt, conf_pat, coop_pat)
            else:
                # 2013-04 through 2013-08 are split daily files
                records = process_daily_month(session, month_dt, conf_pat, coop_pat)
            
            if records:
                df_m = pd.DataFrame(records)
                summary = df_m.groupby("dyad").agg({
                    "total_volume": "sum", "url_conflict_hits": "sum", "url_cooperation_hits": "sum",
                    "cameo_conflict_hits": "sum", "cameo_cooperation_hits": "sum", "goldstein_mean": "mean"
                }).reset_index()
                summary["date"] = date_str
                recovered_records.append(summary)

        # Phase 2: Structural handling of early 90s zero-interaction gaps
        print("\n⚡ Phase 2: Evaluating historical partial missingness gaps (1990-1997)...")
        # Pre-group by year to read the monolithic historical zip only once
        by_year_gaps = {}
        for date_str, missing_dyads in partial_gaps.items():
            year_key = date_str[:4]
            by_year_gaps.setdefault(year_key, []).append((date_str, missing_dyads))

        for year_str, gap_list in tqdm(sorted(by_year_gaps.items()), desc="Historical Dense Balancing"):
            url = f"{GDELT_BASE}{year_str}.zip"
            blob = download_file(session, url)
            
            for date_str, missing_dyads in gap_list:
                target_month_str = date_str.replace("-", "")[:6]
                
                # Check if file downloaded to run a re-extraction look
                records = []
                if blob:
                    records = process_zip(blob, conf_pat, coop_pat, target_month_str, check_date=True)
                
                df_extracted = pd.DataFrame(records) if records else pd.DataFrame()
                
                # Construct zero rows for dyads that truly have zero entries in early GDELT files
                zero_rows = []
                for missing_dyad in missing_dyads:
                    # If it somehow was processed this time, pull the metrics
                    if not df_extracted.empty and missing_dyad in df_extracted["dyad"].values:
                        match = df_extracted[df_extracted["dyad"] == missing_dyad]
                        metrics = match.iloc[0].to_dict()
                    else:
                        # Balanced structural padding
                        metrics = {
                            "dyad": missing_dyad, "total_volume": 0, 
                            "url_conflict_hits": 0, "url_cooperation_hits": 0,
                            "cameo_conflict_hits": 0, "cameo_cooperation_hits": 0, 
                            "goldstein_mean": None
                        }
                    metrics["date"] = date_str
                    zero_rows.append(metrics)
                
                recovered_records.append(pd.DataFrame(zero_rows))

    # Append patches back to primary file safely
    if recovered_records:
        df_patches = pd.concat(recovered_records, ignore_index=True)
        
        # Eliminate rows in original panel that match what we are about to overwrite/patch
        keys_to_remove = set(zip(df_patches["date"], df_patches["dyad"]))
        df_panel_cleaned = df_panel[~df_panel.set_index(["date", "dyad"]).index.isin(keys_to_remove)]
        
        # Merge baseline with corrected elements
        df_final = pd.concat([df_panel_cleaned, df_patches], ignore_index=True)
        
        # Sort values logically by timeline and group to lock structure down
        df_final = df_final.sort_values(by=["date", "dyad"]).reset_index(drop=True)
        df_final = df_final[["date", "dyad", "total_volume", "url_conflict_hits", "url_cooperation_hits", "cameo_conflict_hits", "cameo_cooperation_hits", "goldstein_mean"]]
        
        # Overwrite the master output file with the patched, uniform panel
        df_final.to_csv(OUTPUT_FILE, index=False)
        print(f"\n🎉 Patch injection complete! Balanced panel rewritten back into: {OUTPUT_FILE}")
    else:
        print("\nNo data adjustments were needed.")

if __name__ == "__main__":
    main()

GDELT TARGETED RECOVERY PATCHER
⚡ Phase 1: Executing complete recovery of 2013 transitional months...


2013 Recovery: 100%|██████████| 6/6 [30:31<00:00, 305.29s/it]



⚡ Phase 2: Evaluating historical partial missingness gaps (1990-1997)...


Historical Dense Balancing: 100%|██████████| 8/8 [30:40<00:00, 230.12s/it]
C:\Users\HP\AppData\Local\Temp\ipykernel_7796\1932588202.py:283: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat([df_panel_cleaned, df_patches], ignore_index=True)



🎉 Patch injection complete! Balanced panel rewritten back into: data_nlp\gdelt_birectional_lexical_panel.csv


In [ ]:
import pandas as pd
from pathlib import Path

csv_path = Path("data_nlp/gdelt_birectional_lexical_panel.csv")

print("=" * 60)
print("COMPREHENSIVE GDELT PANEL INTEGRITY AUDIT")
print("=" * 60)

if not csv_path.exists():
    print(f"CRITICAL ERROR: {csv_path} does not exist!")
else:
    df = pd.read_csv(csv_path)
    print(f"-> Total Rows Found: {len(df)}")
    print(f"-> Target Rows for 11 Dyads (1990-2022): {385 * 11} (4,235 expected)")
    
    # Check data distribution by year
    df['date'] = pd.to_datetime(df['date'])
    df['year'] = df['date'].dt.year
    
    print("\n[Row Count and Total Volume Summary by Year]:")
    yearly_summary = df.groupby('year').agg(
        recorded_months=('date', 'nunique'),
        total_rows=('dyad', 'count'),
        sum_volume=('total_volume', 'sum')
    )
    print(yearly_summary)
    
    # Check for empty post-2013 volume
    post_2013 = df[df['year'] >= 2013]
    empty_months = post_2013[post_2013['total_volume'] == 0]
    print(f"\n-> Post-2013 rows with ZERO event volume: {len(empty_months)} / {len(post_2013)}")

print("=" * 60)

COMPREHENSIVE GDELT PANEL INTEGRITY AUDIT
-> Total Rows Found: 4246
-> Target Rows for 11 Dyads (1990-2022): 4235 (4,235 expected)

[Row Count and Total Volume Summary by Year]:
      recorded_months  total_rows  sum_volume
year                                         
1990               12         132        4479
1991               12         132        7535
1992               12         132        6419
1993               12         132        6151
1994               12         132        9811
1995               12         132       14723
1996               12         132       25463
1997               12         132       35498
1998               12         132       31716
1999               12         132       36314
2000               12         132       25021
2001               12         132       32854
2002               12         132       22469
2003               12         132       36929
2004               12         132       30904
2005               12         132       